# Datos
Se crea un archivo que contenga ambos conjuntos de datos y en formato `.feather` por su eficiencia al guardar muchos datos más rápidamenhte que `.csv`.

* Conjunto 1: Datos extraídos de reportes de CENAPRED mediante web scraping `00_web_scraping.ipynb`

  y etiquetados manualmente utilizando una interfaz gráfica `02_interfaz.ipynb`.


* Conjunto 2: Datos obtenidos de videos de youtube mediante:

    ```
    yt-dlp -f 'bestvideo[ext=mp4]' -o 'video.mp4' <URL>
    mkdir frames
    ffmpeg -i video.mp4 -vf "fps=60/60" frames/%06d.jpg
    ```
    
    y etiquetados secuencialmente a simple vista, debido a que las clases suelen ser temporalmente  similares entre imágenes continuas.

## Instalar e importar librerías necesarias

In [1]:
#!pip install numpy
#!pip install pandas
#!pip install tqdm

In [2]:
#!pip install --upgrade huggingface_hub
#!pip install ipywidgets

In [3]:
import os
import numpy as np
import pandas as pd
import tqdm

In [4]:
from tqdm import tqdm

## Importar datos etiquetados manualmente

In [5]:
v2root = "../../popocatepetl-data-v2/"

In [6]:
datav2df = pd.read_feather(v2root + "data.feather")

In [7]:
datav2df["labels"] = datav2df["labels"].str.lower()

In [8]:
datav2df["image"] = v2root + datav2df["image"]

In [9]:
datav2df.columns = ["path", "label"]

In [10]:
# corrección/actualización de etiquetas 
label_mapping = {
    "fumarola": "vapor",
    "erupcion": "ceniza",
    "inactivo": "sin_emision",
    "no visible": "no visible"
}

In [11]:
datav2df["label"] = datav2df["label"].map(label_mapping).fillna(datav2df["label"])

In [12]:
datav2df["id"] = np.full(datav2df.shape[0], -1)

In [13]:
datav2df["video"] = datav2df["path"].str.replace(v2root + "images/", "").replace(".jpg", "")

In [14]:
datav2df

,path,label,id,video
0,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131991.jpg
1,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131992.jpg
2,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131993.jpg
3,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131994.jpg
4,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131995.jpg
...,...,...,...,...
16332,../../popocatepetl-data-v2/images/2024/image_9...,ceniza,-1,2024/image_94.jpg
16333,../../popocatepetl-data-v2/images/2024/image_9...,vapor,-1,2024/image_95.jpg
16334,../../popocatepetl-data-v2/images/2024/image_9...,vapor,-1,2024/image_97.jpg
16335,../../popocatepetl-data-v2/images/2024/image_9...,vapor,-1,2024/image_98.jpg


In [15]:
image_root = '../../popocatepetl-data-v3/'
data=[]
for (dirpath, dirnames, filenames) in tqdm(os.walk(image_root)):
    for file in filenames:
        if file.endswith(".jpg"):
            vpath, label=os.path.split(dirpath)
           
            label = label.replace('_noche', '')
            label = label.replace("erupcion", "ceniza")
            label = label.replace("fumarola", "vapor")
            label = label.replace("inactivo", "sin_emision")
            
            _, vid=os.path.split(vpath)
            id_, _=os.path.splitext(file)
            data.append(pd.Series({
                "path":os.path.join(dirpath, file).replace("\\","/"),
                "label":label,
                "video":vid,
                "id":id_,
            }))
data=pd.DataFrame(data)

181it [00:15, 11.49it/s]


In [16]:
data["id"]=data["id"].astype(int)

In [17]:
data

,path,label,video,id
0,../../popocatepetl-data-v3/v1/no_visible/00459...,no_visible,v1,4590
1,../../popocatepetl-data-v3/v1/no_visible/00459...,no_visible,v1,4591
2,../../popocatepetl-data-v3/v1/no_visible/00459...,no_visible,v1,4592
3,../../popocatepetl-data-v3/v1/no_visible/00459...,no_visible,v1,4593
4,../../popocatepetl-data-v3/v1/no_visible/00459...,no_visible,v1,4594
...,...,...,...,...
224090,../../popocatepetl-data-v3/v9/ceniza/000018.jpg,ceniza,v9,18
224091,../../popocatepetl-data-v3/v9/ceniza/000019.jpg,ceniza,v9,19
224092,../../popocatepetl-data-v3/v9/ceniza/000020.jpg,ceniza,v9,20
224093,../../popocatepetl-data-v3/v9/vapor/000001.jpg,vapor,v9,1


In [18]:
data_merged=pd.concat([datav2df,data])
data=data_merged

In [19]:
data["label"].value_counts()

label
vapor          92387
no_visible     92122
ceniza         32626
sin_emision    23257
Name: count, dtype: int64

In [20]:
data

,path,label,id,video
0,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131991.jpg
1,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131992.jpg
2,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131993.jpg
3,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131994.jpg
4,../../popocatepetl-data-v2/images/1999/ene/p01...,vapor,-1,1999/ene/p0131995.jpg
...,...,...,...,...
224090,../../popocatepetl-data-v3/v9/ceniza/000018.jpg,ceniza,18,v9
224091,../../popocatepetl-data-v3/v9/ceniza/000019.jpg,ceniza,19,v9
224092,../../popocatepetl-data-v3/v9/ceniza/000020.jpg,ceniza,20,v9
224093,../../popocatepetl-data-v3/v9/vapor/000001.jpg,vapor,1,v9


In [21]:
#TO-DO: find sequential pictures, remove edges of sequence prop. to length

In [22]:
data.to_feather("../../popocatepetl-data-v3/data_map.feather")

## Conversión de datos a formato Hugging Face

In [23]:
#!pip install datasets
#!pip install pillow
#!pip install torch
#!pip install torchvision

In [24]:
from datasets import Dataset, Image as dsImage, Value as scalar, ClassLabel, Features, DatasetInfo
from PIL import Image
import torchvision

In [25]:
data["image"]=data["path"]
data.drop(columns=["path"],inplace=True)

In [26]:
class_names=data["label"].unique().tolist()
class_names

['vapor', 'ceniza', 'sin_emision', 'no_visible']

In [27]:
freq=data["label"].value_counts()
freq

label
vapor          92387
no_visible     92122
ceniza         32626
sin_emision    23257
Name: count, dtype: int64

In [28]:
mx=np.max(freq)/freq
mx=mx/mx.sum()
mx

label
vapor          0.113546
no_visible     0.113872
ceniza         0.321528
sin_emision    0.451054
Name: count, dtype: float64

In [29]:
print(mx[class_names].tolist())

[0.11354586666416572, 0.321527676806911, 0.45105396153855953, 0.11387249499036363]


In [30]:
features=Features({
    "label":ClassLabel(num_classes=len(class_names), names=class_names),
    "image":dsImage(),
    "video":scalar("string"),
    "id":scalar("int32"),
})

In [31]:
info=DatasetInfo(
description = "A dataset on Popocatépetl, Mexico's most notorious volcano.",
homepage = "https://github.com/mirluvams",
#license = "MIT",
features = features,
version = "3.0.0",
)

In [40]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 240392 entries, 0 to 224094
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   240392 non-null  object
 1   id      240392 non-null  int64 
 2   video   240392 non-null  object
 3   image   240392 non-null  object
dtypes: int64(1), object(3)
memory usage: 9.2+ MB


In [41]:
dataset=Dataset.from_pandas(data, features=features, preserve_index=False)

In [42]:
dataset=dataset.train_test_split(test_size=.1, seed=36918, stratify_by_column="label")

In [43]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 216352
    })
    test: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 24040
    })
})

In [36]:
dataset.save_to_disk("../dataset")

Saving the dataset (0/11 shards):   0%|          | 0/216352 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/24040 [00:00<?, ? examples/s]

In [37]:
dataset.push_to_hub("mirluvams/popocatepetl", private=True)

Uploading the dataset shards:   0%|          | 0/11 [00:00<?, ? shards/s]

Map:   0%|          | 0/19669 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Map:   0%|          | 0/19669 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

...

Map:   0%|          | 0/12020 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Map:   0%|          | 0/12020 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/mirluvams/popocatepetl/commit/df77c35c9e03cda387c6726f64fd77dd64c85591', commit_message='Upload dataset', commit_description='', oid='df77c35c9e03cda387c6726f64fd77dd64c85591', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mirluvams/popocatepetl', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mirluvams/popocatepetl'), pr_revision=None, pr_num=None)

## Análisis exploratorio de datos

In [54]:
# muestra de imágenes y sus tamaños
train_ds = dataset["train"]

sample_train = train_ds[0]["image"]
print(type(sample_train))   # PIL.Image.Image
print(sample_train.size, sample_train.mode)
#sample_train.show()

<class 'PIL.JpegImagePlugin.JpegImageFile'>
(640, 480) RGB


In [55]:
test_ds = dataset["test"]

sample_test = test_ds[0]["image"]
print(type(sample_test))
print(sample_test.size, sample_test.mode)
#sample_test.show()

<class 'PIL.JpegImagePlugin.JpegImageFile'>
(640, 480) RGB


In [59]:
# análisis general 
sizes = [train_ds[i]["image"].size for i in range(min(100, len(train_ds)))]
widths, heights = zip(*sizes)

print(f"Tamaño promedio: {np.mean(widths):.0f}x{np.mean(heights):.0f}")

Tamaño promedio: 801x559


In [60]:
sizes = [test_ds[i]["image"].size for i in range(min(100, len(test_ds)))]
widths, heights = zip(*sizes)

print(f"Tamaño promedio: {np.mean(widths):.0f}x{np.mean(heights):.0f}")

Tamaño promedio: 840x584


In [74]:
# etiquetas y balance de clases
from collections import Counter

train_labels = [train_ds.features["label"].int2str(i) for i in train_ds["label"]]
test_labels  = [test_ds.features["label"].int2str(i) for i in test_ds["label"]]

train_counts = Counter(train_labels)
test_counts  = Counter(test_labels)

df_counts = pd.DataFrame({
    "train": train_counts,
    "test": test_counts
})

df_counts

,train,test
vapor,83148,9239
no_visible,82910,9212
ceniza,29363,3263
sin_emision,20931,2326
